# Milestone 3 Lab, notebook 1 of 6: Where we are, and the problems by hand

## Where we are

| Notebook | What you do |
|---|---|
| **1. Where we are, and the problems by hand** (you are here) | Recall the question, find the data problems yourself, and correct the answer key |
| 2. Checks in code | Write two quality checks as queries, and prove each one works |
| 3. Quarantine, and the figures before and after | Set the wrong records aside, recalculate, and confirm the totals add up |
| 4. A to-do list and a helper, by hand | Ask a model to explain the problems, then build an agent that organises its own work |
| 5. A harness works the queue | Give every problem to a ready-made agent harness, and score its reports |
| 6. The same question, with real warnings | Ask the analyst the leader's question, now that the warnings come from real checks |

Model requests in this notebook: **0**. A full run of this lab costs about 10 US cents.

## What this notebook is for

In milestone 2 the profit figures became real: they are calculated from the business's records. One thing is still not real. The **warnings** the analyst gives ("three orders appear twice") come from a list that a person wrote by hand.

A hand-written list has two weaknesses. It goes out of date, and it changes nothing: a problem on the list is still counted in the figures.

In this notebook you look for the problems yourself, with no AI. You will find more than the list knew about. Then you work out what one figure should really be.

> If you have not done milestones 1 and 2, do them first.

In [1]:
import pandas as pd
from lab_helpers import load_result, load_source_tables, save_result, show_doc

## Step 1: What the business is asking for now

Here is this milestone's requirement, from the **BRD**.

In [2]:
show_doc("BRD.md", "1. Business problem")

> **From `docs/BRD.md`: 1. Business problem**

After milestone 2, Kestrel Ridge Outfitters calculates its profit from its own records. But nobody checks those records.

The warnings the analyst gives ("three orders appear twice") come from a list a person wrote by hand. If that person misses a problem, the analyst reports full confidence in a wrong figure. And a problem that is known is still counted: the trail water bottle's October profit includes three order lines twice.

The business needs three things:

- **The system finds data problems itself**, every time the figures are calculated.
- **Each problem has a fixed consequence.** Records that are certainly wrong are left out of the figures. Records that are doubtful stay in, with a warning, until a person has looked.
- **Someone has to investigate each problem.** That is slow, repetitive work. The business wants AI to do the first pass and hand a person a short report.

The leader's question has not changed. This is the answer key from milestone 2. Look at the first line: the trail water bottle, $5,188.20. By the end of this notebook you will know that figure is too high.

In [3]:
old_key = load_result("answer_key_milestone_2")

pd.DataFrame(old_key["figures"])[["what", "entity_id", "kind", "value"]]

,what,entity_id,kind,value
0,#1 product: Trail water bottle,P001,value,5188.20
1,#2 product: Headlamp,P008,value,4250.00
2,#3 product: Rain jacket,P009,value,3841.00
3,Top category: Hydration,Hydration,value,13358.06
4,"Hydration, change since the month before",Hydration,change,376.06
5,"Apparel, change since the month before (the bi...",Apparel,change,-5914.87
6,"Camp, change since the month before (cannot be...",Camp,change,NaN
7,Camp stove (its cost is missing),P007,value,NaN


## Step 2: Load the records

These are the same seven tables as in milestone 2. Since then, a few more problems have crept into them, and nobody has updated the hand-written list.

In [4]:
tables = load_source_tables()

lines = tables["order_items"]
orders = tables["orders"]
refunds = tables["refunds"]

print(len(lines), "order lines,", len(orders), "orders,", len(refunds), "refunds")

2549 order lines, 2450 orders, 50 refunds


## Step 3: Look for records that appear twice

pandas can mark rows that are exact copies of another row. `duplicated(keep=False)` marks **every** copy, including the first.

In [5]:
is_a_copy = lines.duplicated(keep=False)
copied_lines = lines[is_a_copy]

print(len(copied_lines), "rows are copies of another row")
copied_lines

14 rows are copies of another row


,order_item_id,order_id,product_id,quantity,unit_list_price,discount_amount
1245,OI-01246,ORD-21207,P001,1,40.0,4.00
1246,OI-01246,ORD-21207,P001,1,40.0,4.00
1433,OI-01433,ORD-21389,P002,1,35.0,1.75
1434,OI-01433,ORD-21389,P002,1,35.0,1.75
1435,OI-01434,ORD-21390,P002,1,35.0,1.75
1436,OI-01434,ORD-21390,P002,1,35.0,1.75
1438,OI-01436,ORD-21392,P002,3,35.0,5.25
1439,OI-01436,ORD-21392,P002,3,35.0,5.25
1440,OI-01437,ORD-21393,P002,4,35.0,7.00
1441,OI-01437,ORD-21393,P002,4,35.0,7.00


The hand-written list said three trail water bottle (P001) lines appear twice. That is here: six rows. But there are also eight rows for the insulated flask (P002) that the list never mentioned.

When were these orders placed? Add the order date to each line.

In [6]:
copied_with_dates = copied_lines.merge(orders, on="order_id")
one_row_each = copied_with_dates[["order_item_id", "product_id", "order_date"]].drop_duplicates()

one_row_each

,order_item_id,product_id,order_date
0,OI-01246,P001,2026-10-01
2,OI-01433,P002,2026-10-05
4,OI-01434,P002,2026-10-05
6,OI-01436,P002,2026-10-05
8,OI-01437,P002,2026-10-05
10,OI-01750,P001,2026-10-13
12,OI-01924,P001,2026-10-17


Look at the dates. The three bottle lines are from three different days. The four flask lines are all from the **same day**. Those look like two different causes. Keep that thought for notebooks 4 and 5.

## Step 4: Look for a product with no cost

A product that was sold in a month needs a cost for that month. First list which products were sold in which months. Cancelled orders do not count as sales.

In [7]:
completed_orders = orders[orders["status"] == "completed"]
sold_lines = lines.merge(completed_orders, on="order_id")
sold_lines["month"] = sold_lines["order_date"].dt.strftime("%Y-%m")
sold = sold_lines[["product_id", "month"]].drop_duplicates()

print(len(sold), "product-months had sales")

24 product-months had sales


Now attach each one's cost. `how="left"` keeps every row of `sold`, and leaves the cost empty where there is none.

In [8]:
sold_with_cost = sold.merge(tables["product_costs"], on=["product_id", "month"], how="left")
no_cost = sold_with_cost[sold_with_cost["unit_cost"].isna()]

no_cost

,product_id,month,unit_cost
17,P007,2026-10,NaN


The camp stove (P007) has no cost for October. The list knew about that one.

## Step 5: Look for a refund larger than its order line

A line's value is quantity × list price, less its discount. A refund should never be more than that.

One trap: some lines appear twice. If we join refunds to lines as they are, a refund on a copied line would appear twice too. So take one of each line first.

In [9]:
one_of_each_line = lines.drop_duplicates().copy()
one_of_each_line["line_value"] = (
    one_of_each_line["quantity"] * one_of_each_line["unit_list_price"] - one_of_each_line["discount_amount"]
)

refunds_with_value = refunds.merge(one_of_each_line[["order_item_id", "line_value"]], on="order_item_id")
too_large = refunds_with_value[refunds_with_value["refund_amount"] > refunds_with_value["line_value"] + 0.005]

too_large

,refund_id,order_item_id,product_id,refund_date,refund_amount,return_handling_cost,line_value
1,REF-0002,OI-00080,P011,2026-09-07,60.0,2.66,25.0


A $60.00 refund on a sun hat (P011) line worth $25.00. The list knew about that one too.

So by hand we found the three problems on the list, and one the list did not have. Are there others we did not think to look for? In notebook 2 the checks will answer that.

## Step 6: What should the trail water bottle's profit be?

The three extra bottle lines are counted in milestone 2's figure. Work out how much profit they add.

`duplicated()` with no argument marks only the copies **after the first**. Those are the extra ones.

In [10]:
extra_lines = lines[lines.duplicated() & (lines["product_id"] == "P001")]

extra_lines

,order_item_id,order_id,product_id,quantity,unit_list_price,discount_amount
1246,OI-01246,ORD-21207,P001,1,40.0,4.0
1755,OI-01750,ORD-21695,P001,1,40.0,4.0
1930,OI-01924,ORD-21860,P001,1,40.0,4.0


Each extra line added sales. Net sales for a line is quantity × list price, less the discount.

In [11]:
extra_net_sales = (extra_lines["quantity"] * extra_lines["unit_list_price"] - extra_lines["discount_amount"]).sum()

print("Net sales counted twice:", extra_net_sales)

Net sales counted twice: 108.0


Each extra line also added costs: the product's cost, a shipment, and a payment fee. The shipments and fees of these lines appear twice as well.

In [12]:
costs = tables["product_costs"]
bottle_cost = costs[(costs["product_id"] == "P001") & (costs["month"] == "2026-10")]["unit_cost"].iloc[0]
extra_product_cost = extra_lines["quantity"].sum() * bottle_cost

shipments = tables["shipments"]
extra_shipments = shipments[shipments.duplicated() & (shipments["product_id"] == "P001")]
extra_delivery = extra_shipments["pick_pack_cost"].sum() + extra_shipments["carrier_cost"].sum()

fees = tables["payment_fees"]
extra_fees = fees[fees.duplicated() & (fees["product_id"] == "P001")]
extra_fee_total = extra_fees["fee_amount"].sum()

print("Product cost counted twice:", extra_product_cost)
print("Delivery cost counted twice:", extra_delivery)
print("Payment fees counted twice: ", extra_fee_total)

Product cost counted twice: 54.0
Delivery cost counted twice: 19.5
Payment fees counted twice:  3.24


Contribution profit is net sales less those costs. So this is the profit that should not be there:

In [13]:
overstated = round(extra_net_sales - extra_product_cost - extra_delivery - extra_fee_total, 2)

print("Contribution profit overstated by:", overstated)
print("Trail water bottle, October, corrected:", round(5188.20 - overstated, 2))

Contribution profit overstated by: 31.26
Trail water bottle, October, corrected: 5156.94


## Step 7: The corrected answer key

Three lines of the answer key include the bottle: the bottle itself, the Hydration category total, and Hydration's change since September. Take the overstated amount off each.

(The flask's copies are new in the records. Milestone 2's key never included them, so its flask figure is already right.)

In [14]:
answer_key = {"figures": [], "confidence": old_key["confidence"]}

for line in old_key["figures"]:
    new_line = dict(line)
    if line["entity_id"] in ("P001", "Hydration"):
        new_line["value"] = round(line["value"] - overstated, 2)
    answer_key["figures"].append(new_line)

save_result("answer_key", answer_key)
pd.DataFrame(answer_key["figures"])[["what", "entity_id", "kind", "value"]]

Saved results/answer_key.json


,what,entity_id,kind,value
0,#1 product: Trail water bottle,P001,value,5156.94
1,#2 product: Headlamp,P008,value,4250.00
2,#3 product: Rain jacket,P009,value,3841.00
3,Top category: Hydration,Hydration,value,13326.80
4,"Hydration, change since the month before",Hydration,change,344.80
5,"Apparel, change since the month before (the bi...",Apparel,change,-5914.87
6,"Camp, change since the month before (cannot be...",Camp,change,NaN
7,Camp stove (its cost is missing),P007,value,NaN


This is the answer key for the rest of this lab. The pipeline you build must produce these figures, and it can only do that if it finds the duplicates and leaves them out.

## What we now know

1. The hand-written list was out of date. Looking at the records found duplicates it did not have.
2. A problem on a list is still in the figures. The bottle's profit was overstated by $31.26.
3. Finding problems by hand is slow, and you only find what you think to look for.
4. We have a corrected answer key to judge the rest of the lab.

**Next:** notebook 2, where each of these searches becomes a check that code runs every time.